# Python meets FSL: checking an environment

A working note about the boundary between a Python notebook and a native neuroimaging tool. Before trusting a larger analysis, I want a small example whose answer I already know.

Neurodesk makes versioned scientific tools available through container-backed modules. This notebook can also run in my smaller, locked comp-env image. These are distinct environments; passing the same check does not make them identical. See `reproduce.md` for the actual tested versions and any compatibility limitations.

We use synthetic data only. No scans, patient information, credentials, or network requests belong in this example. Dependencies are installed outside the notebook.

In [ ]:
import hashlib
import importlib.metadata
import os
import shutil
import sys
from pathlib import Path
import subprocess
import tempfile
import nibabel as nib
import numpy as np

print("Python:", sys.version, sys.executable)
for package in ("numpy", "nibabel", "ipykernel"):
    print(package, importlib.metadata.version(package))
tool = shutil.which("fslmaths")
if tool is None:
    raise RuntimeError("fslmaths is unavailable: select the field-notes neuro image or load the documented Neurodesk FSL module before starting the kernel.")
identity = os.environ.get("FIELD_NOTES_NATIVE_IDENTITY")
if not identity:
    raise RuntimeError("Native identity is missing: use the documented verified launch environment.")
print("Native tool:", tool)
print("Recorded native identity:", identity)
print("The recorded identity is provenance supplied by the launcher, not an independent binary attestation.")

## A volume with a known answer

The array has 512 values (0 through 511), arranged as an 8 × 8 × 8 volume. A diagonal affine gives each voxel 2 mm spacing. This is not an anatomical orientation tutorial: a synthetic array has no anatomical landmarks.

FSL doubles the voxel values. Python independently checks the result, including geometry, with absolute tolerance 1e-6 and zero relative tolerance. A SHA256 of the little-endian float32 array identifies the input without relying on compressed-file timestamps.

In [ ]:
data = np.arange(512, dtype=np.float32).reshape(8, 8, 8)
affine = np.diag([2.0, 2.0, 2.0, 1.0])
print("Input array SHA256:", hashlib.sha256(data.astype("<f4").tobytes(order="C")).hexdigest())
with tempfile.TemporaryDirectory(prefix="field-note-") as directory:
    source = Path(directory) / "input.nii.gz"
    target = Path(directory) / "doubled.nii.gz"
    nib.save(nib.Nifti1Image(data, affine), source)
    try:
        subprocess.run([tool, str(source), "-mul", "2", str(target), "-odt", "float"],
                       check=True, timeout=60, capture_output=True, text=True)
    except subprocess.CalledProcessError as error:
        raise RuntimeError(f"fslmaths failed (exit {error.returncode}): {error.stderr[-2000:]}") from error
    except subprocess.TimeoutExpired as error:
        raise RuntimeError("fslmaths exceeded the 60-second limit") from error
    result = nib.load(target)
    values = result.get_fdata()
    assert result.shape == (8, 8, 8)
    assert np.isfinite(values).all()
    np.testing.assert_allclose(values, data * 2, rtol=0, atol=1e-6)
    np.testing.assert_allclose(result.affine, affine, rtol=0, atol=1e-6)
    np.testing.assert_allclose(result.header.get_zooms(), (2, 2, 2), rtol=0, atol=1e-6)
print("PASS: Python + FSL volume check")

## What did we establish?

A fresh Python kernel could write NIfTI, invoke a native executable, read its output, and recover the expected values and geometry. Temporary images are removed when the block ends.

This does **not** validate preprocessing, registration, a statistical model, clinical use, or arbitrary DICOM inputs. It is a small executable check of the environment boundary.

Try changing the multiplier and updating the independent expected calculation. Restart the kernel and run all cells. Save a copy of your notebook if you want to keep edits; Binder sessions are temporary. Use Run on Binder for the pinned hosted environment, or a compatible local Neurodesk environment.